# 01 — Exploratory Data Analysis

Exploring the IEEE-CIS Fraud Detection dataset: class imbalance, fraud dynamics over time,
missing values and key features. Raw data rows are not shown in the committed outputs
(competition rules forbid redistribution).

In [5]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Python: 3.12.15
pandas: 3.0.6
numpy: 2.5.4


## 1. Raw data overview

In [13]:
RAW_DIR = Path("../data/raw")

start = time.perf_counter()
transactions = pd.read_csv(RAW_DIR / "train_transaction.csv")
identity = pd.read_csv(RAW_DIR / "train_identity.csv")
elapsed = time.perf_counter() - start

print(f"Loaded in {elapsed:.1f} s")
print(f"transactions: {transactions.shape[0]:,} rows x {transactions.shape[1]} columns")
print(f"identity:     {identity.shape[0]:,} rows x {identity.shape[1]} columns")

Loaded in 12.7 s
transactions: 590,540 rows x 394 columns
identity:     144,233 rows x 41 columns


In [14]:
def memory_mb(df):
    """Return the DataFrame memory footprint in megabytes."""
    return df.memory_usage(deep=True).sum() / 1024 ** 2

print(f"transactions: {memory_mb(transactions):,.0f} MB")
print(f"identity:     {memory_mb(identity):,.0f} MB")

transactions: 1,792 MB
identity:     57 MB


In [15]:
transactions.dtypes.value_counts()

float64    376
str         14
int64        4
Name: count, dtype: int64

In [16]:
has_identity = transactions["TransactionID"].isin(identity["TransactionID"])
print(f"Transactions with identity info: {has_identity.mean():.1%}")

Transactions with identity info: 24.4%


## 2. Merging tables and reducing memory

In [17]:
train = transactions.merge(identity, on="TransactionID", how="left", validate="one_to_one")
del transactions, identity

print(f"train: {train.shape[0]:,} rows x {train.shape[1]} columns")
print(f"memory: {memory_mb(train):,.0f} MB")

train: 590,540 rows x 434 columns
memory: 1,984 MB
